In [ ]:
import os, sys, warnings, json, gc, time, traceback
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except Exception as e:
    optuna = None
    print(f'[INFO] optuna import 실패 → 이미 찾은 best params만 사용합니다: {e}')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
warnings.filterwarnings('ignore')

# Kaggle 경로 설정
INPUT_DIR = '/kaggle/input/datasets/min2602/my-data/'
OUTPUT_DIR = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TARGET = '임신 성공 여부'
N_SPLITS = 5

# 시간이 정말 급하면 True 유지. 여유 있으면 False로 바꾸세요.
FAST_MODE = True
N_JOBS = max(1, min(4, os.cpu_count() or 2))

# GPU 가용 여부 확인
def check_gpu():
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print('[GPU] 사용 가능')
            print(result.stdout.split('\n')[0])
            return True
    except Exception:
        pass
    print('[GPU] 사용 불가 — CPU 모드')
    return False

USE_GPU = check_gpu()
print(f'FAST_MODE={FAST_MODE}, N_SPLITS={N_SPLITS}, N_JOBS={N_JOBS}')


In [ ]:
# ── 나이 구간 룩업 (하드코딩 → 데이터 통계 불필요)
AGE_INFO = {
    '만18-34세': {'median': 26.0,  'risk': 'Normal'},
    '만35-37세': {'median': 36.0,  'risk': 'High_Early'},
    '만38-39세': {'median': 38.5,  'risk': 'High_Early'},
    '만40-42세': {'median': 41.0,  'risk': 'High_Extreme'},
    '만43-44세': {'median': 43.5,  'risk': 'High_Extreme'},
    '만45-50세': {'median': 47.5,  'risk': 'High_Reversal'},
    '알 수 없음': {'median': None, 'risk': 'Unknown'},
}
AGE_MEDIAN_FILLNA = 38.5

COLS_TO_DROP = [
    'ID', '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일',
    '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 농도', '불임 원인 - 정자 형태',
    'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부',
]

# 결측 플래그 생성 대상 (반드시 fillna 전 실행)
# '난자 해동 경과일'은 COLS_TO_DROP에 있으므로 여기서 제외
ISNA_FLAG_COLS = ['배아 해동 경과일', '난자 혼합 경과일', '배아 이식 경과일']

# IQR 클리핑 대상 (상한값은 각 fold train에서 계산)
IQR_CLIP_COLS = ['저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수']

KNOWN_REASONS = ['현재 시술용', '배아 저장용', '난자 저장용', '기증용', '연구용']

print('상수 정의 완료')


In [ ]:
train_raw = pd.read_csv(INPUT_DIR + 'train.csv')
test_raw  = pd.read_csv(INPUT_DIR + 'test.csv')
print(f'train: {train_raw.shape}, test: {test_raw.shape}')
print(f'타겟 분포: {train_raw[TARGET].value_counts(normalize=True).round(3).to_dict()}')


In [ ]:
class FoldPreprocessor:
    """
    IQR 클리핑 상한값과 OHE를 fold의 train 데이터에서만 fit하고
    valid / test에는 transform만 적용하는 전처리기.
    """

    def __init__(self):
        self.iqr_bounds = {}    # {col: upper_bound}
        self.ohe = None

    def fit(self, df_train: pd.DataFrame):
        # IQR 클리핑 상한값 계산
        for col in IQR_CLIP_COLS:
            if col in df_train.columns:
                q1 = df_train[col].quantile(0.25)
                q3 = df_train[col].quantile(0.75)
                self.iqr_bounds[col] = q3 + 1.5 * (q3 - q1)

        # OHE fit
        if '특정 시술 유형' in df_train.columns:
            series = df_train['특정 시술 유형'].fillna('').apply(self._classify_treatment)
            try:
                self.ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first', dtype=np.float32)
            except TypeError:
                # sklearn 구버전 대응
                self.ohe = OneHotEncoder(sparse=False, handle_unknown='ignore', drop='first', dtype=np.float32)
            self.ohe.fit(series.to_frame(name='시술_분류_그룹'))
        return self

    @staticmethod
    def _classify_treatment(x):
        t = str(x).upper().strip()
        if 'BLASTOCYST' in t: return 'Blastocyst_Transfer'
        elif 'ICSI' in t:     return 'ICSI'
        elif 'IVF'  in t:     return 'IVF'
        elif 'IUI'  in t:     return 'IUI'
        else:                 return 'Unknown'

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(columns=COLS_TO_DROP, errors='ignore').copy()

        # STEP 1. isna 플래그 (반드시 fillna 전)
        for col in ISNA_FLAG_COLS:
            if col in df.columns:
                df[f'{col}_isna'] = df[col].isna().astype(np.int8)
        # day3_4 ablation: both 모드 — 두 플래그 모두 유지
        if '배아 이식 경과일_isna' in df.columns:
            df['배아이식_없음'] = df['배아 이식 경과일_isna']

        # STEP 2. 유전검사 통합
        df['유전검사_시행여부'] = np.int8(0)  # COLS_TO_DROP에서 이미 제거됨

        # STEP 3. 나이 피처 (하드코딩 룩업)
        if '시술 당시 나이' in df.columns:
            df['Age_Median']     = df['시술 당시 나이'].map(
                lambda x: AGE_INFO.get(x, {'median': None})['median'])
            df['Age_Risk_Group'] = df['시술 당시 나이'].map(
                lambda x: AGE_INFO.get(x, {'risk': 'Unknown'})['risk'])
            df = df.drop(columns=['시술 당시 나이'])

        # STEP 4. 배아 품질 파생변수
        total    = df.get('총 생성 배아 수', pd.Series(0, index=df.index))
        transfer = df.get('이식된 배아 수',  pd.Series(0, index=df.index))
        df['배아_잉여율'] = np.where(total > 0, (total - transfer) / total, 0).clip(0, 1)

        age_med = df.get('Age_Median', pd.Series(AGE_MEDIAN_FILLNA, index=df.index)).fillna(AGE_MEDIAN_FILLNA)
        df['Age×배아수'] = age_med * total

        collected = df.get('수집된 신선 난자 수', pd.Series(0, index=df.index))
        mixed     = df.get('혼합된 난자 수',      pd.Series(0, index=df.index))
        df['수정률'] = np.where(collected > 0, mixed / collected, 0).clip(0, 1)

        stored = df.get('저장된 배아 수', pd.Series(0, index=df.index))
        df['저장배아_보유여부'] = (stored > 0).astype(int)

        def parse_count(col_name):
            s = df.get(col_name, pd.Series('0회', index=df.index))
            return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)
        df['총_시술횟수'] = parse_count('IVF 시술 횟수') + parse_count('DI 시술 횟수')

        df['기증정자_사용여부'] = df.get(
            '기증자 정자와 혼합된 난자 수',
            pd.Series(np.nan, index=df.index)).notna().astype(int)

        # STEP 5. 기증난자 교호작용
        if '난자 출처' in df.columns:
            df['기증난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
        else:
            df['기증난자_여부'] = 0
        df['기증난자×나이'] = df['기증난자_여부'] * age_med
        df['자가난자×나이'] = (1 - df['기증난자_여부']) * age_med

        # STEP 6. 고위험군 교호작용
        age_risk = df.get('Age_Risk_Group', pd.Series('Unknown', index=df.index))
        is_extreme = age_risk.isin(['High_Extreme', 'High_Reversal'])
        is_normal  = age_risk.eq('Normal')
        thaw_isna  = df.get('배아 해동 경과일_isna', pd.Series(1, index=df.index))
        df['초고위험_기증난자_조합'] = (is_extreme & (df['기증난자_여부'] == 1)).astype(np.int8)
        df['고령_동결배아_조합']    = (is_extreme & (thaw_isna == 0)).astype(np.int8)
        df['정상군_첫시술']         = (is_normal  & (df['총_시술횟수'] == 0)).astype(np.int8)

        # STEP 7. 목적 멀티라벨 분리
        if '배아 생성 주요 이유' in df.columns:
            split_series = df['배아 생성 주요 이유'].fillna('').apply(
                lambda x: [v.strip() for v in x.split(',') if v.strip()])
            mlb = MultiLabelBinarizer(classes=KNOWN_REASONS)
            encoded = pd.DataFrame(
                mlb.fit_transform(split_series),
                columns=[f'목적_{c}' for c in KNOWN_REASONS],
                index=df.index)
            df = pd.concat([df.drop(columns=['배아 생성 주요 이유']), encoded], axis=1)

        # STEP 8. OHE (fit에서 저장한 encoder 사용)
        if self.ohe is not None and '특정 시술 유형' in df.columns:
            series = df['특정 시술 유형'].fillna('').apply(self._classify_treatment)
            ohe_arr  = self.ohe.transform(series.to_frame(name='시술_분류_그룹'))
            try:
                feature_names = self.ohe.get_feature_names_out(['시술_분류_그룹'])
            except AttributeError:
                feature_names = self.ohe.get_feature_names(['시술_분류_그룹'])
            ohe_cols = [f'시술_{c}' for c in feature_names]
            ohe_df   = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
            df = pd.concat([df.drop(columns=['특정 시술 유형']), ohe_df], axis=1)

        # STEP 9. IQR 클리핑 (fit에서 저장한 상한값 사용)
        for col, upper in self.iqr_bounds.items():
            if col in df.columns:
                df[col] = df[col].clip(upper=upper)

        # STEP 10. 범주형 결측 → '알 수 없음'
        cat_cols = df.select_dtypes(include='object').columns
        df[cat_cols] = df[cat_cols].fillna('알 수 없음')

        # STEP 11. 수치형 결측 → -999 (트리 구조적 결측 학습)
        num_cols = df.select_dtypes(include=[np.number]).columns
        df[num_cols] = df[num_cols].fillna(-999)

        return df

print('FoldPreprocessor 정의 완료')


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3모델 OOF 함수 — Kaggle P100 긴급 안정화 버전
# 핵심:
# 1) LGBM: object → pandas category로 고정
# 2) CatBoost: categorical → str, numeric → float32로 고정
# 3) XGBoost: pandas DataFrame을 DMatrix에 직접 넣지 않고 np.float32 matrix로 변환
# ─────────────────────────────────────────────────────────────

FILL_NUM = -999.0
FILL_CAT = '알 수 없음'
UNK_CAT  = '__UNK__'


def _elapsed(t0):
    return f'{time.time() - t0:.1f}s'


def _align_columns(X_tr, X_val, X_te, fill_value=FILL_NUM):
    """fold train 컬럼 순서를 기준으로 valid/test 컬럼을 강제 정렬."""
    cols = list(X_tr.columns)
    X_val = X_val.reindex(columns=cols, fill_value=fill_value)
    X_te  = X_te.reindex(columns=cols, fill_value=fill_value)
    return X_tr.copy(), X_val.copy(), X_te.copy()


def _is_cat_series(s):
    return (s.dtype == object) or pd.api.types.is_categorical_dtype(s)


def _numeric_np(s, fill_value=FILL_NUM):
    arr = pd.to_numeric(s, errors='coerce').astype('float32').to_numpy(copy=True)
    arr = np.nan_to_num(arr, nan=fill_value, posinf=fill_value, neginf=fill_value)
    return arr.astype(np.float32, copy=False)


def _prepare_lgbm_dfs(X_tr, X_val, X_te):
    """LightGBM용: 문자열 컬럼을 category dtype으로 변환."""
    X_tr, X_val, X_te = _align_columns(X_tr, X_val, X_te)
    cat_cols = [c for c in X_tr.columns if _is_cat_series(X_tr[c])]

    for col in cat_cols:
        tr_s = X_tr[col].where(X_tr[col].notna(), FILL_CAT).astype(str)
        cats = list(pd.Index(tr_s.unique()).astype(str))
        if UNK_CAT not in cats:
            cats.append(UNK_CAT)

        for X in (X_tr, X_val, X_te):
            s = X[col].where(X[col].notna(), FILL_CAT).astype(str)
            s = s.where(s.isin(cats), UNK_CAT)
            X[col] = pd.Categorical(s, categories=cats)

    # 나머지는 float32로 고정
    for col in X_tr.columns:
        if col not in cat_cols:
            X_tr[col]  = _numeric_np(X_tr[col])
            X_val[col] = _numeric_np(X_val[col])
            X_te[col]  = _numeric_np(X_te[col])

    return X_tr, X_val, X_te, cat_cols


def _prepare_catboost_dfs(X_tr, X_val, X_te):
    """CatBoost용: categorical은 문자열, numeric은 float32."""
    X_tr, X_val, X_te = _align_columns(X_tr, X_val, X_te)
    cat_cols = [c for c in X_tr.columns if _is_cat_series(X_tr[c])]

    for col in cat_cols:
        for X in (X_tr, X_val, X_te):
            X[col] = X[col].where(X[col].notna(), FILL_CAT).astype(str)

    for col in X_tr.columns:
        if col not in cat_cols:
            X_tr[col]  = _numeric_np(X_tr[col])
            X_val[col] = _numeric_np(X_val[col])
            X_te[col]  = _numeric_np(X_te[col])

    return X_tr, X_val, X_te, cat_cols


def _xgb_to_numpy(df_tr, df_val, df_te):
    """
    XGBoost용 완전 안전 변환.
    pandas DataFrame 대입/카테고리 dtype을 모두 우회하고, 최종 np.float32 matrix만 DMatrix에 전달.
    """
    df_tr, df_val, df_te = _align_columns(df_tr, df_val, df_te)
    tr_cols, val_cols, te_cols = [], [], []

    for col in df_tr.columns:
        if pd.api.types.is_numeric_dtype(df_tr[col]):
            tr_cols.append(_numeric_np(df_tr[col]))
            val_cols.append(_numeric_np(df_val[col]))
            te_cols.append(_numeric_np(df_te[col]))
        else:
            tr_s  = df_tr[col].where(df_tr[col].notna(), FILL_CAT).astype(str)
            val_s = df_val[col].where(df_val[col].notna(), FILL_CAT).astype(str)
            te_s  = df_te[col].where(df_te[col].notna(), FILL_CAT).astype(str)

            # train fold에 존재한 category만 mapping. valid/test unseen은 -1.
            uniques = list(pd.Index(tr_s.unique()).astype(str))
            mapper = {v: i for i, v in enumerate(uniques)}
            tr_cols.append(tr_s.map(mapper).fillna(-1).astype('float32').to_numpy())
            val_cols.append(val_s.map(mapper).fillna(-1).astype('float32').to_numpy())
            te_cols.append(te_s.map(mapper).fillna(-1).astype('float32').to_numpy())

    Xtr = np.column_stack(tr_cols).astype(np.float32, copy=False)
    Xva = np.column_stack(val_cols).astype(np.float32, copy=False)
    Xte = np.column_stack(te_cols).astype(np.float32, copy=False)
    return Xtr, Xva, Xte


def _make_fold_data(train_raw, test_raw, y, tr_idx, val_idx, model_type):
    preprocessor = FoldPreprocessor()
    preprocessor.fit(train_raw.iloc[tr_idx])

    X_tr  = preprocessor.transform(train_raw.iloc[tr_idx]).drop(columns=[TARGET], errors='ignore')
    X_val = preprocessor.transform(train_raw.iloc[val_idx]).drop(columns=[TARGET], errors='ignore')
    X_te  = preprocessor.transform(test_raw).drop(columns=[TARGET], errors='ignore')

    if model_type == 'lgbm':
        return (*_prepare_lgbm_dfs(X_tr, X_val, X_te), y.iloc[tr_idx], y.iloc[val_idx])
    if model_type == 'catboost':
        return (*_prepare_catboost_dfs(X_tr, X_val, X_te), y.iloc[tr_idx], y.iloc[val_idx])
    if model_type == 'xgb':
        X_tr, X_val, X_te = _xgb_to_numpy(X_tr, X_val, X_te)
        return X_tr, X_val, X_te, None, y.iloc[tr_idx], y.iloc[val_idx]
    raise ValueError(model_type)


# ── LightGBM OOF
def run_lgbm_oof(train_raw, test_raw, y, params, n_splits=N_SPLITS, tag='lgbm'):
    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    full_params = {
        **params,
        'scale_pos_weight': scale_pos_weight,
        'objective': 'binary',
        'metric': 'auc',
        'random_state': RANDOM_STATE,
        'n_jobs': globals().get('N_JOBS', -1),
        'verbose': -1,
    }
    if USE_GPU:
        full_params.update({'device_type': 'gpu', 'max_bin': 63})

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(y), dtype=np.float32)
    test_preds = np.zeros(len(test_raw), dtype=np.float32)
    t_all = time.time()

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_raw, y), 1):
        t0 = time.time()
        X_tr, X_val, X_te, cat_feats, y_tr, y_val = _make_fold_data(
            train_raw, test_raw, y, tr_idx, val_idx, 'lgbm')

        model = lgb.LGBMClassifier(**full_params)
        try:
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
                categorical_feature=cat_feats if cat_feats else 'auto',
            )
        except Exception as e:
            if USE_GPU and full_params.get('device_type') == 'gpu':
                print(f'  [{tag}] Fold {fold}: LGBM GPU 실패 → CPU fallback: {str(e)[:120]}')
                cpu_params = {k: v for k, v in full_params.items() if k not in ['device_type', 'gpu_use_dp']}
                cpu_params.pop('max_bin', None)
                model = lgb.LGBMClassifier(**cpu_params)
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
                    categorical_feature=cat_feats if cat_feats else 'auto',
                )
            else:
                raise

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1].astype(np.float32) / n_splits
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        print(f'  [{tag}] Fold {fold}/{n_splits} AUC: {fold_auc:.5f} | best_iter={getattr(model, "best_iteration_", None)} | {_elapsed(t0)}')
        del X_tr, X_val, X_te, model
        gc.collect()

    auc = roc_auc_score(y, oof_preds)
    print(f'[{tag}] OOF AUC: {auc:.5f} | total {_elapsed(t_all)}')
    return oof_preds, test_preds, auc


# ── CatBoost OOF
def run_catboost_oof(train_raw, test_raw, y, params=None, n_splits=N_SPLITS, tag='catboost'):
    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    default_params = {
        'iterations': 2000,
        'learning_rate': 0.05,
        'depth': 6,
        'l2_leaf_reg': 3,
        'early_stopping_rounds': 100,
        'eval_metric': 'AUC',
        'random_seed': RANDOM_STATE,
        'scale_pos_weight': scale_pos_weight,
        'verbose': 0,
        'allow_writing_files': False,
        'thread_count': globals().get('N_JOBS', -1),
    }
    if USE_GPU:
        default_params.update({'task_type': 'GPU', 'devices': '0'})

    full_params = {**default_params, **(params or {})}
    if USE_GPU:
        full_params['task_type'] = 'GPU'
        full_params['devices'] = '0'
        # class_weights와 scale_pos_weight 동시 사용 방지
        full_params.pop('class_weights', None)
        full_params['scale_pos_weight'] = scale_pos_weight

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(y), dtype=np.float32)
    test_preds = np.zeros(len(test_raw), dtype=np.float32)
    t_all = time.time()

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_raw, y), 1):
        t0 = time.time()
        X_tr, X_val, X_te, cat_feats, y_tr, y_val = _make_fold_data(
            train_raw, test_raw, y, tr_idx, val_idx, 'catboost')

        model = CatBoostClassifier(**full_params)
        try:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_feats, verbose=0)
        except Exception as e:
            if USE_GPU and full_params.get('task_type') == 'GPU':
                print(f'  [{tag}] Fold {fold}: CatBoost GPU 실패 → CPU fallback: {str(e)[:120]}')
                cpu_params = {k: v for k, v in full_params.items() if k not in ['task_type', 'devices']}
                model = CatBoostClassifier(**cpu_params)
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_feats, verbose=0)
            else:
                raise

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1].astype(np.float32) / n_splits
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        best_iter = model.get_best_iteration() if hasattr(model, 'get_best_iteration') else None
        print(f'  [{tag}] Fold {fold}/{n_splits} AUC: {fold_auc:.5f} | best_iter={best_iter} | {_elapsed(t0)}')
        del X_tr, X_val, X_te, model
        gc.collect()

    auc = roc_auc_score(y, oof_preds)
    print(f'[{tag}] OOF AUC: {auc:.5f} | total {_elapsed(t_all)}')
    return oof_preds, test_preds, auc


def _xgb_train_with_fallback(params, dtrain, dval, num_boost_round, early_stopping_rounds):
    evals = [(dval, 'val')]
    try:
        return xgb.train(
            params, dtrain,
            num_boost_round=num_boost_round,
            evals=evals,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False,
        )
    except Exception as e:
        if USE_GPU and params.get('device') == 'cuda':
            print(f'    XGB device=cuda 실패 → gpu_hist 시도: {str(e)[:100]}')
            p2 = params.copy()
            p2.pop('device', None)
            p2['tree_method'] = 'gpu_hist'
            p2['predictor'] = 'gpu_predictor'
            try:
                return xgb.train(
                    p2, dtrain,
                    num_boost_round=num_boost_round,
                    evals=evals,
                    early_stopping_rounds=early_stopping_rounds,
                    verbose_eval=False,
                )
            except Exception as e2:
                print(f'    XGB gpu_hist 실패 → CPU hist fallback: {str(e2)[:100]}')
                p3 = params.copy()
                p3.pop('device', None)
                p3.pop('predictor', None)
                p3['tree_method'] = 'hist'
                return xgb.train(
                    p3, dtrain,
                    num_boost_round=num_boost_round,
                    evals=evals,
                    early_stopping_rounds=early_stopping_rounds,
                    verbose_eval=False,
                )
        raise


def _xgb_predict_best(model, dmat):
    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        try:
            return model.predict(dmat, iteration_range=(0, best_iter + 1))
        except TypeError:
            best_ntree = getattr(model, 'best_ntree_limit', 0)
            if best_ntree:
                return model.predict(dmat, ntree_limit=best_ntree)
    return model.predict(dmat)


# ── XGBoost OOF
def run_xgb_oof(train_raw, test_raw, y, params, n_splits=N_SPLITS, tag='xgb'):
    """
    day3_4 결론 반영:
    - include_log_clip=False: _log/_clip 파생 피처 추가 없음
    - transfer_flag_mode='both': 배아이식_없음 + 배아 이식 경과일_isna 둘 다 유지
    """
    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    full_params = {
        **params,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'scale_pos_weight': scale_pos_weight,
        'seed': RANDOM_STATE,
        'verbosity': 0,
        'tree_method': 'hist',
        'max_bin': 256,
        'nthread': globals().get('N_JOBS', -1),
    }
    if USE_GPU:
        full_params['device'] = 'cuda'

    num_boost_round = globals().get('XGB_NUM_BOOST_ROUND', 2200 if FAST_MODE else 3000)
    early_stop = globals().get('XGB_EARLY_STOPPING', 80 if FAST_MODE else 100)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(y), dtype=np.float32)
    test_preds = np.zeros(len(test_raw), dtype=np.float32)
    t_all = time.time()

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_raw, y), 1):
        t0 = time.time()
        X_tr, X_val, X_te, _, y_tr, y_val = _make_fold_data(
            train_raw, test_raw, y, tr_idx, val_idx, 'xgb')

        dtrain = xgb.DMatrix(X_tr,  label=y_tr.to_numpy())
        dval   = xgb.DMatrix(X_val, label=y_val.to_numpy())
        dte    = xgb.DMatrix(X_te)

        model = _xgb_train_with_fallback(full_params, dtrain, dval, num_boost_round, early_stop)
        oof_preds[val_idx] = _xgb_predict_best(model, dval)
        test_preds += _xgb_predict_best(model, dte).astype(np.float32) / n_splits
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        print(f'  [{tag}] Fold {fold}/{n_splits} AUC: {fold_auc:.5f} | best_iter={getattr(model, "best_iteration", None)} | {_elapsed(t0)}')
        del X_tr, X_val, X_te, dtrain, dval, dte, model
        gc.collect()

    auc = roc_auc_score(y, oof_preds)
    print(f'[{tag}] OOF AUC: {auc:.5f} | total {_elapsed(t_all)}')
    return oof_preds, test_preds, auc

print('OOF 학습 함수 3개 정의 완료 — dtype 안정화 / GPU fallback / XGB np.float32 변환 적용')


In [ ]:
y = train_raw[TARGET].astype(int)

# LightGBM: v5 best params (day2_2)
LGBM_PARAMS = {
    'learning_rate':     0.01267950928343402,
    'num_leaves':        16,
    'min_child_samples': 42,
    'subsample':         0.5274814138166641,
    'subsample_freq':    1,
    'colsample_bytree':  0.9656063914879106,
    'reg_alpha':         1.1739806261228718,
    'reg_lambda':        6.158190762189439,
    'n_estimators':      3000,
}

# CatBoost: Optuna best params (새 Optuna 탐색은 시간상 돌리지 않음)
CATBOOST_PARAMS = {
    'depth': 6,
    'learning_rate': 0.015305744365500184,
    'l2_leaf_reg': 10,
    'iterations': 2500,
    'bagging_temperature': 0.9394989415641891,
    'random_strength': 1.7896547008552977,
    'eval_metric': 'AUC',
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'early_stopping_rounds': 100,
}

# XGBoost: day3_3 Optuna best params + day3_4 ablation 결론(no_log_clip, both_flags)
XGB_PARAMS = {
    'max_depth':        4,
    'learning_rate':    0.04051765582302794,
    'min_child_weight': 4,
    'subsample':        0.9045886131253115,
    'colsample_bytree': 0.5000089943404621,
    'gamma':            3.670343132433689,
    'reg_alpha':        8.38442030937844,
    'reg_lambda':       5.898413746507709,
}

# P100 시간 절약 설정. 성능이 더 필요하고 시간이 있으면 FAST_MODE=False로 바꾸고 재실행.
if FAST_MODE:
    LGBM_PARAMS['n_estimators'] = min(LGBM_PARAMS['n_estimators'], 2200)
    CATBOOST_PARAMS['iterations'] = min(CATBOOST_PARAMS['iterations'], 1800)
    CATBOOST_PARAMS['early_stopping_rounds'] = 80
    XGB_NUM_BOOST_ROUND = 2200
    XGB_EARLY_STOPPING = 80
else:
    XGB_NUM_BOOST_ROUND = 3000
    XGB_EARLY_STOPPING = 100

print('파라미터 정의 완료')
print(f'  FAST_MODE={FAST_MODE}')
print(f'  LGBM: num_leaves={LGBM_PARAMS["num_leaves"]}, lr={LGBM_PARAMS["learning_rate"]:.5f}, n_estimators={LGBM_PARAMS["n_estimators"]}')
print(f'  CatBoost: depth={CATBOOST_PARAMS["depth"]}, lr={CATBOOST_PARAMS["learning_rate"]:.5f}, iterations={CATBOOST_PARAMS["iterations"]}')
print(f'  XGBoost: max_depth={XGB_PARAMS["max_depth"]}, lr={XGB_PARAMS["learning_rate"]:.5f}, rounds={XGB_NUM_BOOST_ROUND}')


In [ ]:
def load_or_train_oof(name, runner, *args, **kwargs):
    """한 모델이 끝나면 OOF/test를 저장. 런타임 끊겨도 완료된 모델은 재사용."""
    oof_path  = os.path.join(OUTPUT_DIR, f'oof_{name}.npy')
    test_path = os.path.join(OUTPUT_DIR, f'test_{name}.npy')
    meta_path = os.path.join(OUTPUT_DIR, f'meta_{name}.json')

    if os.path.exists(oof_path) and os.path.exists(test_path):
        oof = np.load(oof_path)
        test = np.load(test_path)
        auc = roc_auc_score(y, oof)
        print(f'[CACHE] {name}: OOF AUC={auc:.5f} 로드 완료')
        return oof, test, auc

    oof, test, auc = runner(*args, **kwargs)
    np.save(oof_path, oof)
    np.save(test_path, test)
    with open(meta_path, 'w') as f:
        json.dump({'name': name, 'auc': float(auc), 'fast_mode': bool(FAST_MODE)}, f, indent=2)
    print(f'[SAVE] {name}: {oof_path}, {test_path}')
    return oof, test, auc


print('=' * 60)
print('  STEP 1/3 — LightGBM (v5 best params)')
print('=' * 60)
lgbm_oof, lgbm_test, lgbm_auc = load_or_train_oof(
    'lgbm_v5', run_lgbm_oof,
    train_raw, test_raw, y, LGBM_PARAMS, tag='lgbm_v5')

print('\n' + '=' * 60)
print('  STEP 2/3 — CatBoost (Optuna best params)')
print('=' * 60)
cat_oof, cat_test, cat_auc = load_or_train_oof(
    'catboost_optuna', run_catboost_oof,
    train_raw, test_raw, y, CATBOOST_PARAMS, tag='catboost_optuna')

print('\n' + '=' * 60)
print('  STEP 3/3 — XGBoost (no_log_clip, both_flags)')
print('=' * 60)
xgb_oof, xgb_test, xgb_auc = load_or_train_oof(
    'xgb_no_log_clip_both_flags', run_xgb_oof,
    train_raw, test_raw, y, XGB_PARAMS, tag='xgb_no_log_clip_both_flags')

print('\n단일 모델 OOF 요약')
print(f'  LGBM     : {lgbm_auc:.5f}')
print(f'  CatBoost : {cat_auc:.5f}')
print(f'  XGBoost  : {xgb_auc:.5f}')


In [ ]:
def find_best_weights_3model(oof_list, test_list, y, labels, step=0.01):
    best_auc, best_w, best_test = 0, None, None
    results = []

    for w1 in np.arange(0.0, 1.01, step):
        for w2 in np.arange(0.0, 1.01 - w1, step):
            w3 = round(1.0 - round(w1, 4) - round(w2, 4), 4)
            if w3 < -0.001: continue
            w3 = max(w3, 0.0)
            blend = w1*oof_list[0] + w2*oof_list[1] + w3*oof_list[2]
            auc   = roc_auc_score(y, blend)
            results.append({'w1': round(w1,4), 'w2': round(w2,4), 'w3': w3, 'auc': round(auc, 6)})
            if auc > best_auc:
                best_auc = auc
                best_w   = (round(w1,4), round(w2,4), w3)
                best_test = w1*test_list[0] + w2*test_list[1] + w3*test_list[2]

    results_df = pd.DataFrame(results).sort_values('auc', ascending=False)
    print(f'\n[3모델 앙상블] Top 5 가중치 조합:')
    print(results_df.head(5).to_string(index=False))
    print(f'\n최적 가중치: {labels[0]}={best_w[0]:.2f}, '
          f'{labels[1]}={best_w[1]:.2f}, {labels[2]}={best_w[2]:.2f}')
    print(f'앙상블 OOF AUC: {best_auc:.4f}')
    return best_test, best_w, best_auc, results_df


def find_best_weights_2model(oof_list, test_list, y, labels, step=0.01):
    best_auc, best_w, best_test = 0, None, None
    for w1 in np.arange(0.0, 1.01, step):
        w2  = round(1.0 - w1, 4)
        auc = roc_auc_score(y, w1*oof_list[0] + w2*oof_list[1])
        if auc > best_auc:
            best_auc = auc
            best_w   = (round(w1,4), w2)
            best_test = w1*test_list[0] + w2*test_list[1]
    print(f'[{labels[0]}+{labels[1]}] OOF AUC: {best_auc:.4f}  가중치: {best_w}')
    return best_test, best_w, best_auc


# 비교 실험
print('─' * 50)
print('[비교 1] LGBM + CatBoost (기존 Day2 앙상블 재현)')
t_lc, w_lc, auc_lc = find_best_weights_2model(
    [lgbm_oof, cat_oof], [lgbm_test, cat_test], y, ['LGBM', 'CatBoost'])

print('\n[비교 2] LGBM + XGBoost')
t_lx, w_lx, auc_lx = find_best_weights_2model(
    [lgbm_oof, xgb_oof], [lgbm_test, xgb_test], y, ['LGBM', 'XGB'])

print('\n[비교 3] CatBoost + XGBoost')
t_cx, w_cx, auc_cx = find_best_weights_2model(
    [cat_oof, xgb_oof], [cat_test, xgb_test], y, ['CatBoost', 'XGB'])

print('\n[비교 4] LGBM + CatBoost + XGBoost (3모델)')
t_3, w_3, auc_3, results_df = find_best_weights_3model(
    [lgbm_oof, cat_oof, xgb_oof],
    [lgbm_test, cat_test, xgb_test],
    y, labels=['LGBM', 'CatBoost', 'XGB'])


In [ ]:
print('=' * 65)
print('             Day 3-5 최종 결과 요약')
print('=' * 65)
print(f'  [기준] Day1 v4 앙상블 LB:          0.74191')
print(f'  [기준] Day2 v5 앙상블 OOF:         0.7403')
print(f'  LGBM (v5 params) OOF:              {lgbm_auc:.5f}')
print(f'  CatBoost (Optuna) OOF:             {cat_auc:.5f}')
print(f'  XGBoost OOF:                       {xgb_auc:.5f}')
print(f'  LGBM + CatBoost OOF:               {auc_lc:.5f}  가중치: {w_lc}')
print(f'  LGBM + XGBoost OOF:                {auc_lx:.5f}  가중치: {w_lx}')
print(f'  CatBoost + XGBoost OOF:            {auc_cx:.5f}  가중치: {w_cx}')
print(f'  LGBM + CatBoost + XGBoost OOF:     {auc_3:.5f}  가중치: {w_3}')
print('=' * 65)

# 최고 조합 자동 선택
candidates = [
    ('LGBM+CB',  auc_lc, t_lc),
    ('LGBM+XGB', auc_lx, t_lx),
    ('CB+XGB',   auc_cx, t_cx),
    ('3model',   auc_3,  t_3),
]
best_name, best_auc_final, best_test_final = max(candidates, key=lambda x: x[1])
print(f'\n✅ 최종 채택: {best_name}  OOF AUC: {best_auc_final:.5f}')

# 제출 파일 저장 — sample_submission의 예측 컬럼명을 자동 감지
sample_sub = pd.read_csv(INPUT_DIR + 'sample_submission.csv')
pred_col = 'probability' if 'probability' in sample_sub.columns else sample_sub.columns[-1]
sample_sub[pred_col] = best_test_final
safe_name = best_name.replace('+', '_')
save_path = os.path.join(OUTPUT_DIR, f'submission_day3_5_{safe_name}.csv')
sample_sub.to_csv(save_path, index=False)
print(f'저장: {save_path}')

# 단일 모델 제출 파일도 저장
single_preds = {
    'lgbm': lgbm_test,
    'catboost': cat_test,
    'xgb_no_log_clip_both_flags': xgb_test,
}
for name, preds in single_preds.items():
    s = pd.read_csv(INPUT_DIR + 'sample_submission.csv')
    s[pred_col] = preds
    s.to_csv(os.path.join(OUTPUT_DIR, f'submission_day3_5_{name}.csv'), index=False)

# OOF 결과표 저장
summary_df = pd.DataFrame([
    {'model': 'lgbm', 'oof_auc': lgbm_auc},
    {'model': 'catboost', 'oof_auc': cat_auc},
    {'model': 'xgb_no_log_clip_both_flags', 'oof_auc': xgb_auc},
    {'model': 'lgbm_catboost', 'oof_auc': auc_lc, 'weights': str(w_lc)},
    {'model': 'lgbm_xgb', 'oof_auc': auc_lx, 'weights': str(w_lx)},
    {'model': 'catboost_xgb', 'oof_auc': auc_cx, 'weights': str(w_cx)},
    {'model': 'lgbm_catboost_xgb', 'oof_auc': auc_3, 'weights': str(w_3)},
    {'model': f'BEST_{best_name}', 'oof_auc': best_auc_final},
])
summary_path = os.path.join(OUTPUT_DIR, 'day3_5_oof_summary.csv')
summary_df.to_csv(summary_path, index=False)

print('단일 모델 제출 파일도 저장 완료.')
print('\n=== 생성/수정된 파일 목록 ===')
print(f'생성: {save_path}')
print('생성: submission_day3_5_lgbm.csv')
print('생성: submission_day3_5_catboost.csv')
print('생성: submission_day3_5_xgb_no_log_clip_both_flags.csv')
print(f'생성: {summary_path}')
